# Search the Bundestag energy papers

Type a question, press **Search**, and you get the text passages from Bundestag papers that match it best, with a link to the PDF.

**How to use:** run all cells once (menu *Run → Run All*). In VS Code, pick the `.venv` Python as kernel (top right). Loading takes about 10 seconds. After that, you can search as often as you like.

**First time only:** build the index (see the scripts folder) and start the notebook from the project folder:

```
.venv/bin/pip install -r requirements-notebook.txt
.venv/bin/jupyter notebook notebooks/search.ipynb
```

In [ ]:
# Load the search index and the embedding model. Takes about 10 seconds.
import os
import sys
from pathlib import Path

# Work from the project folder, so "data/chroma" and the code in "src" are found.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

from energiewende.index import search, store

index = store.load_index()
print(store.count_chunks(), "text passages ready to search")

In [ ]:
# The search box.
import html

import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="z.B. Wie soll der Ausbau von Wasserstoff gefördert werden?",
    layout=widgets.Layout(width="80%"),
)
results_slider = widgets.IntSlider(value=4, min=1, max=10, description="Results")
search_button = widgets.Button(description="Search", button_style="primary")
results_box = widgets.HTML()  # the results are shown here


def card(hit):
    """One result as a small card: title, source, and the matching passage."""
    return f"""
    <div style="border-left: 4px solid #2a7ae2; padding: 4px 12px; margin: 12px 0;">
      <b>{html.escape(hit["title"][:200])}</b><br>
      <small>Drucksache {hit["number"]} · {hit["date"]} · score {hit["score"]} ·
      <a href="{hit["url"]}" target="_blank">PDF</a></small>
      <p style="white-space: pre-wrap;">{html.escape(hit["text"][:700])} …</p>
    </div>"""


def on_search(_):
    question = question_box.value.strip()
    if not question:
        return
    try:
        hits = search.search(index, question, k=results_slider.value)
    except Exception as error:  # show the error instead of hiding it
        results_box.value = f"<pre style='color: #c0392b;'>Error: {html.escape(repr(error))}</pre>"
        return
    results_box.value = "".join(card(hit) for hit in hits)


search_button.on_click(on_search)
display(widgets.HBox([question_box, search_button]), results_slider, results_box)